<a href="https://colab.research.google.com/github/JoviWZhu/20206RAG/blob/RAG-Finetuning/FineWeb_Edu_Fine_Tuning_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers trl peft accelerate bitsandbytes datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 14.8 MB/s eta 0:00:00


In [2]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

from google.colab import userdata
HF_TOKEN_DEV = userdata.get('HF_TOKEN_DEV')

# ==========================================
# 1. HARDWARE & ACCELERATION CONFIGURATION
# ==========================================
MODEL_ID = "google/gemma-2-2b-it"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True
)

print(f"⚡ Step 1: Loading model '{MODEL_ID}' in 4-bit precision...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN_DEV  # <--- PASS THE TOKEN HERE TO UNLOCK THE GATED REPO
)

# Remember to pass it to the tokenizer as well!
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN_DEV  # <--- PASS THE TOKEN HERE AS WELL
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


⚡ Step 1: Loading model 'google/gemma-2-2b-it' in 4-bit precision...


config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.0k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

In [6]:
from datasets import Dataset, load_dataset

# ==========================================
# 2. FIXED DATA PREPARATION (Single-Row Mapping)
# ==========================================
print("\n⚡ Step 2: Loading and formatting a slice of FineWeb-Edu...")
# Pull a small sample split of FineWeb-Edu for streaming training data efficiency
dataset = load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", split="train", streaming=True)

# Take 500 samples and transform them into a native Hugging Face Dataset Object
raw_list_sample = list(dataset.take(500))
dataset_sample = Dataset.from_list(raw_list_sample)
print(f"✅ Successfully prepared {len(dataset_sample)} examples as a native Dataset.")

def formatting_prompts_func(example):
    """
    FIXED: Processes ONE row dictionary at a time instead of returning a nested list array.
    This safely allows TRL's internal tokenizers and EOS appenders to process elements.
    """
    # Create the structured prompt for a single record row
    text = (
        f"<bos><start_of_turn>user\nSynthesize an expert educational overview of this technical concept.<end_of_turn>\n"
        f"<start_of_turn>model\n{example['text']}<end_of_turn><eos>"
    )
    # Return a dictionary configuration mapping directly back to the text column identifier
    return {"text": text}



⚡ Step 2: Loading and formatting a slice of FineWeb-Edu...


README.md:   0%|          | 0.00/26.4k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

✅ Successfully prepared 500 examples as a native Dataset.


In [3]:

# ==========================================
# 3. LoRA ADAPTER CONFIGURATION
# ==========================================
print("\n⚡ Step 3: Configuring Parameter-Efficient Fine-Tuning (LoRA)...")
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "o_proj", "k_proj", "v_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)



⚡ Step 3: Configuring Parameter-Efficient Fine-Tuning (LoRA)...


In [4]:
from trl import SFTConfig, SFTTrainer

# ==========================================
# 4. CORRECTED TRAINING CONFIGURATION (SFTConfig)
# ==========================================
training_args = SFTConfig(
    output_dir="./teacher_model_output",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    max_steps=50,
    optim="paged_adamw_8bit",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    save_strategy="steps",
    save_steps=25,
    report_to="none",
    max_length=512,                     # <--- CHANGED FROM max_seq_length TO max_length
    dataset_text_field="text"
)



In [7]:
# ==========================================
# 5. INITIALIZE SFTTRAINER LOOP
# ==========================================
print("\n⚡ Step 5: Initializing Hugging Face SFTTrainer execution stream...")

# We map the dataset transformation ahead of time using our single-row function.
# This bypasses all internal formatting conflicts inside the initialization script!
formatted_dataset = dataset_sample.map(formatting_prompts_func)

trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset, # Pass the completely pre-formatted dataset directly
    peft_config=peft_config,
    processing_class=tokenizer,
    args=training_args                 # Uses your max_length config
)

# Launch the fine-tuning calculation loop
print("\n🚀 Commencing Fine-Tuning Execution!")
trainer.train()



⚡ Step 5: Initializing Hugging Face SFTTrainer execution stream...


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

...

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1}.



🚀 Commencing Fine-Tuning Execution!


Step,Training Loss
10,2.511945
20,2.464561
30,2.320919
40,2.350544
50,2.321902


/usr/local/lib/python3.13/dist-packages/peft/utils/other.py:1496: UserWarning: Unable to fetch remote file due to the following error 401 Client Error. (Request ID: Root=1-6aa37c4e-135c37742a1bb59e4b1dd6bf;4a8f7cae-9fd0-4168-9dce-4837d8c63566)

Cannot access gated repo for url https://huggingface.co/google/gemma-2-2b-it/resolve/main/config.json.
Access to model google/gemma-2-2b-it is restricted. You must have access to it and be authenticated to access it. Please log in. - silently ignoring the lookup for the file config.json in google/gemma-2-2b-it.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/utils/save_and_load.py:438: UserWarning: Could not find a config file in google/gemma-2-2b-it - will assume that the vocabulary was not modified.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/utils/other.py:1496: UserWarning: Unable to fetch remote file due to the following error 401 Client Error. (Request ID: Root=1-6aa37fe9-084554c84ffe46606b11d2c4;f9e9f8e2-c6

TrainOutput(global_step=50, training_loss=2.3939742279052734, metrics={'train_runtime': 1841.5204, 'train_samples_per_second': 0.217, 'train_steps_per_second': 0.027, 'total_flos': 2394087445048320.0, 'train_loss': 2.3939742279052734, 'epoch': 0.8})

In [9]:
# ==========================================
# 6. INFERENCE COMPARISON (VERIFICATION STEP)
# ==========================================
print("\n⚡ Step 6: Testing fine-tuned inference path...")

# Set the model to evaluation mode (turns off dropout layers for deterministic output)
model.eval()

# Construct the exact prompt structure matching your training format
eval_prompt = (
    "<bos><start_of_turn>user\n"
    "Synthesize an expert educational overview of this technical concept: Quantum Computing.\n"
    "<start_of_turn>model\n"
)

# Tokenize and push vectors directly to your T4 GPU memory
inputs = tokenizer(eval_prompt, return_tensors="pt").to("cuda")

print("🤖 Generating text from fine-tuned parameters...")
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=250, # Increased token threshold slightly to let the teacher elaborate
        temperature=0.3,    # Low creativity multiplier ensures structured factual answers
        do_sample=True      # Allows temperature control parameters to operate
    )

print("\n🎓 [FINE-TUNED TEACHER RESPONSE]:")
# FIX: Slices away the prompt tokens so you only print the newly generated text
generated_tokens = outputs[0][inputs.input_ids.shape[1]:]
print(tokenizer.decode(generated_tokens, skip_special_tokens=True))



⚡ Step 6: Testing fine-tuned inference path...
🤖 Generating text from fine-tuned parameters...

🎓 [FINE-TUNED TEACHER RESPONSE]:
Quantum computing is a field of computer science that is concerned with the development of computers that use the principles of quantum mechanics.
Quantum computers are computers that use the principles of quantum mechanics to perform computations. These computers are much faster than classical computers, and they can be used to solve problems that are impossible to solve with classical computers.
Quantum computers are still in their early stages of development, but they have the potential to revolutionize many different fields.
Some of the fields that could be revolutionized by quantum computers include:
- Medicine: Quantum computers could be used to design new drugs and to develop new treatments for diseases.
- Materials science: Quantum computers could be used to design new materials with new properties.
- Finance: Quantum computers could be used to devel